In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()
db_conn.setup_db()

####  Steps to create project
1. Run the next cell.
2. Specify the example either clastic or carbonate.
3. 30-7a.qppp project will be saved in data\04_project folder.

* Note that the required curves in the LAS files are 'GR', 'RT', 'NPHI', 'RHOB'

In [ ]:
data_path = 'data/'
filenames = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        filenames.append(os.path.join(root, file)) if file.endswith('.las') and not 'seismic' in root  else None
print(filenames)

project_name = "22-30"
with db_conn.get_session() as db_session:
    proj = Project(db_session, name=project_name)
    proj.read_las(filenames, depth_uom='m')
    data = proj.get_all_data()        

In [ ]:
# Clean up data
merged_df = data.copy()

merged_df['NPHI'] = np.where(
    merged_df.groupby('WELL_NAME')['NPHI'].transform('mean') > 1,
    merged_df['NPHI'] / 100,
    merged_df['NPHI']
)
# merged_df['GR'] = np.where(merged_df.GR.notna(), merged_df.GR, merged_df.GRI)
merged_df['RHOB'] = np.where(merged_df.RHOB.notna(), merged_df.RHOB, merged_df.DEN)
merged_df['RT'] = np.where(merged_df.RESD.notna(), merged_df.RESD, merged_df.ILD)
merged_df['DTC'] = np.where(merged_df.DTC.notna(), merged_df.DTC, merged_df.DT)
merged_df['CALI'] = np.where(merged_df.CALI.notna(), merged_df.CALI, merged_df.CAL)

In [ ]:
# Process core data
all_core_data = pd.read_excel(r"data\CoreAnalysis_Export_Selected_GBR_22_30.xls")
all_core_data['WELL_NAME'] = all_core_data['Wellname'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')
all_core_data['DEPTH'] = pd.to_numeric(all_core_data['Sample_Depth'], errors='coerce')
all_core_data['DEPTH'] = np.where(all_core_data['Ft_Mtr'] == 'Ft', round(all_core_data['DEPTH'] / 3.281, 4), all_core_data['DEPTH'])
all_core_data['CPORE'] = pd.to_numeric(all_core_data['Porosity_1'], errors='coerce') / 100
all_core_data['CPERM'] = pd.to_numeric(all_core_data['Hor_Permeability_1'], errors='coerce')

cols = ['DEPTH', 'CPORE', 'CPERM', 'CORE_ID']
merged_core_df = pd.DataFrame()
return_df = pd.DataFrame()
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    temp_df = well_data.copy()
    core_data = all_core_data[all_core_data.WELL_NAME == well_name]
    core_data = core_data.sort_values('DEPTH')
    num_core_points = len(core_data)
    core_ids = [str(i) for i in range(1, num_core_points + 1)]
    core_data['CORE_ID'] = core_ids
    merged_core_df = pd.concat([merged_core_df, core_data])

    # Merge with well logs
    temp_df = pd.merge_asof(well_data, core_data[['DEPTH', 'CPORE', 'CPERM', 'CORE_ID']],
                            direction='nearest', on='DEPTH', tolerance=0.1524 / 2)
    return_df = pd.concat([return_df, temp_df], ignore_index=True)
merged_core_df['WELL_NAME'] = merged_core_df['WELL_NAME'].str.replace('-', '/', 1).str.replace('-', ' ')
merged_core_df = merged_core_df[['WELL_NAME'] + cols + [c for c in merged_core_df.columns if c not in cols]]
merged_core_df.to_csv(r'data\merged_core_data.csv', index=False)
merged_df = return_df.copy()

In [ ]:
# Process zones
tops_df = pd.read_excel(r"data\UK_22-30_Tops.xlsx")
tops_df['WELL_NAME'] = tops_df['WELLREGNO'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')

return_df = pd.DataFrame()
for well_name, well_data in tqdm(merged_df.groupby('WELL_NAME'), desc='Merge Tops data'):
    tqdm.write(f'Processing {well_name} well')
    
    # Process sub groups/ formations
    marker_df = tops_df[(tops_df.WELL_NAME == well_name)].copy()
    marker_df['ZONES'] = marker_df['Top_Pick_Name']
    marker_df['DEPTH'] = marker_df['Depth_MD'].astype('float')
    marker_df = marker_df.sort_values('DEPTH').reset_index(drop=True)

    # Merge with well logs
    temp_df = pd.merge_asof(well_data, marker_df[['DEPTH', 'ZONES', 'model']], direction='backward', on='DEPTH')
    
    # Clean No Tops intervals
    temp_df = temp_df[temp_df.ZONES.notna()]
    temp_df = temp_df.sort_values('DEPTH').reset_index(drop=True)
    return_df = pd.concat([return_df, temp_df], ignore_index=True)
merged_df = return_df.copy()

In [ ]:
# Process TVDSS
import wellpathpy as wpp

filenames = os.listdir(r'data')
return_df = pd.DataFrame()
for well_name, well_data in tqdm(merged_df.groupby('WELL_NAME'), desc='Calculating TVD'):
    temp_df = well_data.sort_values('DEPTH').drop_duplicates('DEPTH')
    filename = [f for f in filenames if well_name in f and 'deviation' in f.lower()]
    if filename:
        filepath = os.path.join('data', filename[0])
        data_dict = pd.read_excel(filepath, sheet_name=None, verbose=False)
        survey_key = [k for k in data_dict.keys() if 'Dev' in k][0]
        well_survey = data_dict.get(survey_key)
        well_survey = well_survey.rename(columns={
            'Well Name': 'WELL_NAME', 'MD M': 'md', 'Inclination': 'incl', 'Azimuth': 'azim',
            'Grid_Easting': 'x', 'Grid_Northing': 'y', 'TVD M': 'TVD', 'TVDSS M': 'TVDSS'
            })
        well_survey['WELL_NAME'] = well_survey['WELL_NAME'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')
        coords = well_survey[well_survey.WELL_NAME == well_name].sort_values('md').reset_index(drop=True)
        dev_survey = wpp.deviation(coords['md'], coords['incl'], coords['azim'])
        # Filter temp_df to only the range covered by the survey
        temp_df = temp_df[temp_df.DEPTH.between(coords['md'].min(), coords['md'].max())]
        tvd = dev_survey.minimum_curvature().resample(temp_df.DEPTH.values).depth
        tqdm.write(f'Processing {well_name}: {len(tvd)} tvd data, {len(temp_df)} well data')
        temp_df['TVD'] = tvd
    return_df = pd.concat([return_df, temp_df])
merged_df = return_df.copy()

In [ ]:
from quick_pp.plotter.plotter import plotly_log

# Plot the results
well_name = '22-30b-11'
well_data = merged_df[merged_df.WELL_NAME == well_name]
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))

In [ ]:
from quick_pp.plotter.plotter import plotly_log

# Plot the results
folder = r'data\04_project\nb_outputs\initial_logs'
os.makedirs(folder, exist_ok=True)
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    well_data = merged_df[merged_df.WELL_NAME == well_name]
    fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
    fig.write_html(os.path.join(folder, f'{well_name}_plot.html'), config=dict(scrollZoom=True))

In [ ]:
# Save result to database
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    project.update_data(merged_df)
    project.save()